# Improve the dataset used for SBERT finetuning

The dataset used for finetuning the ModernBERT model for similarity looks strange with respect to the very opinionated similarity values. We can do better by using a (very good) reranking model and calculating the similarities with this.

It turns out that these also improves the performance of the resulting finetuned model!

Be careful, this notebook is not yet compatible with the latest `transformers` library!

In [ ]:
from transformers import AutoModel

Now we need a new package from `sentence_transformers` which can predict the similarity between two sentences.

In [ ]:
model = AutoModel.from_pretrained(
    'jinaai/jina-reranker-v3',
    dtype="auto",
    trust_remote_code=True,
)
model.cuda()

Use the *flawed* dataset again

In [ ]:
from datasets import load_dataset
train_dataset = load_dataset("sentence-transformers/all-nli", "pair-score", split="train")

In [ ]:
df = train_dataset.to_pandas()

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

In [ ]:
df.head(20).style.background_gradient(cmap='coolwarm')

Create a temporary dataframe containing the start of the dataset and some arbitrary samples:

In [ ]:
dft = pd.concat([df.head(20), df.sample(20, random_state=42)])

Create the instruct codes for prompting the LLM:

In [ ]:
pairs = [(s1, s2) 
           for s1, s2 in zip(dft["sentence1"], dft["sentence2"])]

Run the actual reranking process:

In [ ]:
%%time
from tqdm.auto import tqdm
scores = []
for s1, s2 in tqdm(zip(dft["sentence1"], dft["sentence2"]), total=len(dft)):
    scores.append(model.rerank(s1, s2)[0]['relevance_score'])

In [ ]:
# integrate scores
dft["scores"] = scores

In [ ]:
dft.style.background_gradient(cmap='coolwarm')